# Optional: Wie alt ist der Sternhaufen?

> Dieses Notebook ist ein **Zusatz**. Du kannst es direkt nach Notebook 1 bearbeiten, wenn du
> deinen Haufen von Hand ausgewählt hast, oder nach Notebook 2, wenn du weiter mit den
> Algorithmen gearbeitet hast. Es braucht nur eines: eine Auswahl von Sternen, von denen du
> glaubst, dass sie zusammengehören.

Bisher ging es immer um die Frage **welche Sterne** zusammengehören. Jetzt kommt eine andere
dazu: **wie alt** sind sie?

Die Sterne eines Haufens sind alle etwa gleichzeitig aus derselben Gaswolke entstanden. Sie
sind also gleich alt und gleich weit weg — verschieden ist praktisch nur ihre Masse. Und die
Masse entscheidet über alles Weitere: Schwere Sterne sind heiß, blau und hell, leichte Sterne
kühl, rot und schwach. Trägt man Farbe gegen Helligkeit auf, liegen die Sterne eines Haufens
deshalb nicht wild verstreut, sondern auf **einer Linie**. Dieses Bild heißt
Farben-Helligkeits-Diagramm.

Und diese Linie verändert sich mit der Zeit. Schwere Sterne verbrauchen ihren Brennstoff sehr
viel schneller als leichte. Nach einigen Millionen Jahren sind die schwersten schon fertig und
verlassen die Linie am oberen Ende — je älter der Haufen, desto weiter unten bricht sie ab.
Aus der Stelle, an der sie abknickt, lässt sich das Alter ablesen.

Berechnete Modelle solcher Linien heißen **Isochronen** (griechisch für „gleiche Zeit"): eine
Isochrone zeigt, wo Sterne eines bestimmten Alters liegen müssten. Du legst sie über deine
Sterne und siehst nach, welche am besten passt.

## Vorbereitung

Neben deiner Auswahl brauchst du eine zweite Datei: die Isochronen. Sie kommen nicht von Gaia,
sondern von einer Modellrechnung. Lade sie dir unter <https://stev.oapd.inaf.it/cgi-bin/cmd>
herunter und lege sie neben deine Gaia-Datei in den Ordner `data/`.

**Wichtig beim Herunterladen:**

- Stelle das *photometrische System* auf Gaia (Evans et al. 2018) um. Sonst enthält die Datei keine Gaia-Helligkeiten
  und lässt sich nicht mit deinen Sternen vergleichen.
- Gib einen **Altersbereich** an, nicht ein einzelnes Alter. Du willst ja vergleichen.
- Die Ergebnisseite verlinkt die fertige Datei. Klicke den Link mit der rechten Maustaste
  an und wähle „Verknüpfte Datei laden" bzw. „Ziel speichern unter". Kopiere den Text
  **nicht** aus dem Browser heraus.

Aus deiner Gaia-Datei brauchst du außerdem zwei Spalten, die du bisher nicht benutzt hast:
`phot_g_mean_mag`, die gemessene Helligkeit, und `bp_rp`, die Farbe. Falls sie in deinem
Download fehlen, hole ihn mit diesen beiden Spalten noch einmal.

In [ ]:
%matplotlib widget

from stellar_cluster_finder import (
    add_absolute_magnitude,
    get_isochrone,
    load_isochrones,
    load_parquet,
    plot_and_save,
)

sterne = load_parquet("../data/meine_auswahl.parquet")
print(f"{len(sterne)} Sterne geladen")
sterne.head()

Jetzt legst du fest, **welche** Sterne dein Haufen sind. Genau eine der beiden Zeilen brauchst
du — je nachdem, woher du kommst:

In [ ]:
# Nach Notebook 1: deine Handauswahl (eine Spalte mit True/False).
haufen = sterne[sterne["auswahl_gesamt"]]

# Nach Notebook 2: ein Ergebnis der Algorithmen. Alles ab Label 0 gehört zu einem Haufen,
# -1 ist Rauschen. Ersetze den Spaltennamen durch den Durchlauf, den du am besten fandest.
# haufen = sterne[sterne["position_3d_HDBSCAN"] >= 0]

print(f"{len(haufen)} von {len(sterne)} Sternen gehören zu deinem Haufen")

## 1. Von scheinbarer zu wirklicher Helligkeit

Gaia misst, wie hell ein Stern uns **erscheint**. Das hängt aber genauso von der Entfernung ab
wie vom Stern selbst: Eine Kerze in der Hand ist heller als ein Scheinwerfer am Horizont. Eine
Isochrone sagt dagegen, wie hell die Sterne **wirklich** sind.

Um beides vergleichen zu können, muss die Entfernung herausgerechnet werden. Das Ergebnis heißt
absolute Helligkeit: die Helligkeit, die ein Stern hätte, wenn er in 10 Parsec Entfernung stünde.

Dafür brauchst du die Spalte `distance_pc` aus Notebook 1 — die Entfernung, die du dort schon
benutzt hast, um deinen Haufen auszuwählen.

> **Achtung, kleine Falle:** Helligkeiten werden in *Magnituden* gemessen, und die zählen
> rückwärts. Je **kleiner** die Zahl, desto **heller** der Stern. Deshalb steht in den Bildern
> gleich die y-Achse auf dem Kopf.

In [ ]:
haufen = add_absolute_magnitude(haufen)

print(haufen[["phot_g_mean_mag", "distance_pc", "abs_g_mag", "bp_rp"]].describe().round(2))

## 2. Erst ansehen: das Farben-Helligkeits-Diagramm

Wie immer zuerst das Bild ohne alles Weitere. Links stehen die blauen, heißen Sterne, rechts die
roten, kühlen; oben die hellen, unten die schwachen.

In [ ]:
plot_and_save(
    haufen,
    "bp_rp",
    "abs_g_mag",
    title="Dein Haufen im Farben-Helligkeits-Diagramm",
    xlabel="Farbe BP - RP  (links blau, rechts rot)",
    ylabel="absolute Helligkeit G  (oben hell)",
    invert_yaxis=True,
)

Siehst du das schmale Band, das von links oben nach rechts unten läuft? Das ist die
**Hauptreihe** — dort verbringen Sterne den größten Teil ihres Lebens.

Wie sauber ist dein Band? Liegen die Sterne dicht auf einer Linie, oder ist eine breite Wolke
darunter? Diese Wolke sind Sterne, die gar nicht zum Haufen gehören: Sie stehen in Wirklichkeit
woanders, und mit der falschen Entfernung bekommen sie auch die falsche absolute Helligkeit.

**Das ist die ehrlichste Rückmeldung auf deine Auswahl im ganzen Kurs.** Eine gute Auswahl
ergibt eine schmale Linie, eine schlechte eine Wolke. Wenn du unzufrieden bist, geh zurück und
wähle enger aus — es lohnt sich für alles Weitere.

## 3. Die Isochronen dazu

Jetzt die Modelle. `load_isochrones` sagt dir, welche Alter deine Datei enthält.

In [ ]:
isochronen = load_isochrones("../data/pleiades_7-96-02.dat")

print(sorted(isochronen["age_myr"].unique()))

Eine einzelne Isochrone holst du dir mit `get_isochrone`. Ist das gewünschte Alter nicht in der
Datei, nimmt die Funktion das nächstgelegene und sagt dir welches.

Die Isochrone hat dieselben zwei Spalten wie deine Sterne: `bp_rp` für die Farbe und `Gmag` für
die absolute Helligkeit. Deshalb kannst du sie direkt als Linie über dein Bild legen.

In [ ]:
alter = 100  # <-- Regler: Alter in Millionen Jahren

eine_isochrone = get_isochrone(isochronen, alter)

plot_and_save(
    haufen,
    "bp_rp",
    "abs_g_mag",
    title=f"Dein Haufen mit einer Isochrone für {alter} Mio. Jahre",
    xlabel="Farbe BP - RP",
    ylabel="absolute Helligkeit G",
    invert_yaxis=True,
    line_x=eine_isochrone["bp_rp"],
    line_y=eine_isochrone["Gmag"],
    line_label=f"{alter} Mio. Jahre",
)

## 4. Verschiedene Alter vergleichen

Ein einzelnes Bild sagt wenig — interessant wird es im Vergleich. Die nächste Zelle zeichnet
mehrere Bilder hintereinander, eines je Alter.

In [ ]:
for alter in [40, 100, 250, 1000, 2500]:  # <-- Regler: erst weit auseinander, dann enger
    eine_isochrone = get_isochrone(isochronen, alter)
    plot_and_save(
        haufen,
        "bp_rp",
        "abs_g_mag",
        title=f"{alter} Millionen Jahre",
        xlabel="Farbe BP - RP",
        ylabel="absolute Helligkeit G",
        invert_yaxis=True,
        line_x=eine_isochrone["bp_rp"],
        line_y=eine_isochrone["Gmag"],
        line_label=f"{alter} Mio. Jahre",
    )

> **Vergleiche die Bilder.** Zwei der Linien sollten offensichtlich nicht passen:
>
> - Die **jüngste** liegt am roten, unteren Ende über deinen Sternen. So junge Sterne haben sich
>   noch gar nicht fertig zusammengezogen und leuchten deshalb heller als die deinen.
> - Die **älteste** knickt schon weit unten ab, mitten in deinem Sternband. Wäre dein Haufen so
>   alt, dürfte es die hellen Sterne oben links gar nicht mehr geben — es gibt sie aber.
>
> Damit hast du dein Alter eingegrenzt, ohne vorher zu wissen, wie alt der Haufen ist. Engere
> Werte einsetzen, bis es nicht mehr passt: Wo liegen deine Grenzen?

> **Und jetzt der ehrliche Teil.** Sieh dir die Linien innerhalb deines Bereichs an. Über weite
> Strecken liegen sie fast genau übereinander — der Unterschied sitzt **oben links, am
> Abknickpunkt**. Genau dort werden Sterne aber selten, und die allerhellsten misst Gaia
> schlecht, weil sie den Detektor überstrahlen.
>
> - Wie viele deiner Sterne liegen überhaupt oben am Abknickpunkt?
> - Wie weit kannst du dein Alter dadurch wirklich eingrenzen — auf einen Faktor 2? Auf 10 %?
> - Falls du beide Haufen bearbeitest: Bei welchem der beiden gelingt es dir besser, und woran
>   liegt das? Ein älterer Haufen knickt weiter unten ab, dort wo Gaia gut messen kann.
>
> Eine ehrliche Antwort lautet hier nicht „125 Millionen Jahre", sondern „zwischen X und Y, und
> genauer geht es mit diesen Daten nicht". Auch das ist ein Ergebnis.

## 5. Heranzoomen auf den Abknickpunkt

Der Unterschied zwischen den Altern sitzt oben links — und genau dort ist er im Gesamtbild
untergegangen. Das Diagramm reicht über mehr als ein Dutzend Größenklassen; der Teil, auf den es
ankommt, macht davon nur wenige Prozent aus.

Mit `xlim` und `ylim` schneidest du beide Achsen auf diesen Bereich zu. Die Grenzen liest du
dabei nicht aus der Luft, sondern aus deinen eigenen Sternen: in der Farbe das blaueste Zehntel,
in der Helligkeit das hellste Zehntel, und nach oben ein paar Größenklassen Luft, damit die
Modelllinien noch mit ins Bild passen. So arbeitet die Zelle für jeden Haufen und nicht nur für
einen bestimmten.

In [ ]:
# Der Ausschnitt wird aus deinen eigenen Sternen bestimmt: das blaueste und das hellste Zehntel,
# dazu etwas Luft nach oben für die Modelllinien. Engere Zahlen kannst du danach von Hand
# einsetzen.  # <-- anpassen
ausschnitt_x = (float(haufen["bp_rp"].min()) - 0.2, float(haufen["bp_rp"].quantile(0.10)))
ausschnitt_y = (float(haufen["abs_g_mag"].min()) - 3, float(haufen["abs_g_mag"].quantile(0.10)))

for alter in [100, 250, 1000]:  # <-- Regler: dieselben Alter wie oben, nur herangezoomt
    eine_isochrone = get_isochrone(isochronen, alter)
    plot_and_save(
        haufen,
        "bp_rp",
        "abs_g_mag",
        title=f"Abknickpunkt bei {alter} Mio. Jahren",
        xlabel="Farbe BP - RP",
        ylabel="absolute Helligkeit G",
        invert_yaxis=True,
        xlim=ausschnitt_x,
        ylim=ausschnitt_y,
        line_x=eine_isochrone["bp_rp"],
        line_y=eine_isochrone["Gmag"],
        line_label=f"{alter} Mio. Jahre",
    )

> **Jetzt siehst du das Problem in voller Größe.** Der Abknickpunkt jeder Modelllinie liegt
> deutlich **über** deinen hellsten Sternen: Die Linien biegen dort ab, wo du überhaupt keine
> Messwerte mehr hast.
>
> - Bis zu welcher Helligkeit reichen deine Sterne nach oben, und wo biegt jede Linie ab?
> - Unterhalb dieser Stelle liegen die Linien fast genau übereinander. Woran willst du dann noch
>   entscheiden, welche besser passt?
> - Warum fehlen die hellsten Sterne? Zwei Gründe kommen zusammen: Gaia übersteuert bei sehr
>   hellen Sternen, und von den schwersten Sternen gibt es in einem Haufen ohnehin nur eine
>   Handvoll.
>
> Das ist kein Fehler in deiner Auswahl und keiner in den Modellen — an genau dieser Stelle geben
> die Daten nicht mehr her. Ein weiter Altersbereich lässt sich damit eingrenzen, ein enger
> nicht. Probiere deshalb ruhig Alter, die weit auseinanderliegen: Zwischen 40 und 2500
> Millionen Jahren siehst du einen Unterschied, zwischen 100 und 150 nicht.

## Fragen zum Nachdenken

- Ein Haufen ist über sein Alter definiert worden, weil alle Sterne gleichzeitig entstanden
  sind. Wie gut ist diese Annahme eigentlich? Was würdest du im Diagramm sehen, wenn sie nicht
  stimmt?
- Für die absolute Helligkeit hast du für **jeden** Stern seine eigene Entfernung benutzt. Für
  einen Stern, der gar nicht zum Haufen gehört, ist diese Entfernung trotzdem richtig gemessen
  — warum landet er im Diagramm dann trotzdem an der falschen Stelle?
- Zwischen uns und dem Haufen liegt Staub, der die Sterne schwächer und röter erscheinen lässt.
  In welche Richtung im Diagramm verschiebt das die Sterne? Und würdest du den Haufen dadurch
  für älter oder für jünger halten?
- Die Isochronen sind Rechnungen, keine Messungen. Was müsste in so ein Modell alles hineingehen?
  Und wie würdest du merken, wenn eine der Annahmen darin falsch wäre?
- Du hast jetzt drei ganz verschiedene Wege benutzt, um über denselben Haufen etwas zu erfahren:
  Entfernung, gemeinsame Bewegung und Farbe. Welcher davon hat dich am meisten überzeugt?